In [ ]:
import os
import zipfile
import requests
import pandas as pd
from io import BytesIO
from dotenv import load_dotenv


# 1) INDEXING
# Carregar API key do .env
load_dotenv()

# ============================================================
# PASSO 1: Carregar dados da CVM
# ============================================================

print("=" * 50)
print("PASSO 1: Baixando dados da CVM...")
print("=" * 50)

# Baixar DFP (Demonstrações Financeiras Padronizadas) de 2023
url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_2023.zip"
response = requests.get(url)
z = zipfile.ZipFile(BytesIO(response.content))

# Ver o que tem dentro do ZIP
print("\nArquivos no ZIP:")
for name in z.namelist():
    if "con" in name:  # só consolidados
        print(f"  {name}")

PASSO 1: Baixando dados da CVM...

Arquivos no ZIP:
  dfp_cia_aberta_BPA_con_2023.csv
  dfp_cia_aberta_BPP_con_2023.csv
  dfp_cia_aberta_DFC_MD_con_2023.csv
  dfp_cia_aberta_DFC_MI_con_2023.csv
  dfp_cia_aberta_DMPL_con_2023.csv
  dfp_cia_aberta_DRA_con_2023.csv
  dfp_cia_aberta_DRE_con_2023.csv
  dfp_cia_aberta_DVA_con_2023.csv


In [2]:

# ============================================================
# PASSO 2: Filtrar dados da Petrobras
# ============================================================

print("\n" + "=" * 50)
print("PASSO 2: Filtrando dados da Petrobras...")
print("=" * 50)

# Carregar DRE consolidada
dre = pd.read_csv(
    z.open("dfp_cia_aberta_DRE_con_2023.csv"),
    sep=";",
    encoding="latin-1",
    dtype={"CD_CONTA": str}
)

# Código CVM da Petrobras = 9512
COD_CVM_PETROBRAS = 9512

# Filtrar só Petrobras
dre_petro = dre[dre["CD_CVM"] == COD_CVM_PETROBRAS].copy()

# Filtrar exercício mais recente (ÚLTIMO, não PENÚLTIMO)
dre_petro = dre_petro[dre_petro["ORDEM_EXERC"] == "ÚLTIMO"]

# Ver as colunas disponíveis
print("\nColunas do CSV:")
print(dre_petro.columns.tolist())

# Ver algumas contas
print("\nContas da Petrobras (DRE 2023):")
print(dre_petro[["CD_CONTA", "DS_CONTA", "VL_CONTA"]].head(15).to_string())


PASSO 2: Filtrando dados da Petrobras...

Colunas do CSV:
['CNPJ_CIA', 'DT_REFER', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'GRUPO_DFP', 'MOEDA', 'ESCALA_MOEDA', 'ORDEM_EXERC', 'DT_INI_EXERC', 'DT_FIM_EXERC', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA', 'ST_CONTA_FIXA']

Contas da Petrobras (DRE 2023):
         CD_CONTA                                                DS_CONTA     VL_CONTA
19969        3.01                  Receita de Venda de Bens e/ou Serviços  511994000.0
19971        3.02                   Custo dos Bens e/ou Serviços Vendidos -242061000.0
19973        3.03                                         Resultado Bruto  269933000.0
19975        3.04                          Despesas/Receitas Operacionais  -80591000.0
19977     3.04.01                                     Despesas com Vendas  -25163000.0
19979     3.04.02                       Despesas Gerais e Administrativas   -7952000.0
19981     3.04.03              Perdas pela Não Recuperabilidade de Ativos          0.0
19983     3.04.0

In [3]:
# ============================================================
# PASSO 3: Converter tabela em texto narrativo
# ============================================================

"""
O que estamos fazendo
Transformando linhas de CSV em texto que o LLM consegue entender.
Esse é o passo mais criativo do pipeline.

De onde vem isso
Chip Huyen (Cap. 6 — RAG Beyond Texts): Dados não-textuais
(tabelas, imagens, áudio) precisam ser convertidos em texto ou
ter embeddings especializados. Para dados financeiros tabulares,
a conversão em texto narrativo é a abordagem mais prática.
"""

print("\n" + "=" * 50)
print("PASSO 3: Convertendo tabela em texto...")
print("=" * 50)

# Contas que importam para risco de crédito
CONTAS_CREDITO = {
    "3.01": "Receita Líquida",
    "3.05": "Resultado Operacional (EBIT)",
    "3.06": "Resultado Financeiro",
    "3.06.02": "Despesas Financeiras",
    "3.11": "Lucro/Prejuízo do Período",
}

# Filtrar só contas relevantes
dre_credito = dre_petro[dre_petro["CD_CONTA"].isin(CONTAS_CREDITO.keys())]

# Converter em texto narrativo
linhas_texto = ["Petrobras (PETR4) — Demonstração do Resultado — 2023:", ""]

for _, row in dre_credito.iterrows():
    valor = row["VL_CONTA"]
    conta = row["DS_CONTA"]
    linhas_texto.append(f"  {conta}: R$ {valor:,.0f} mil")

texto_dre = "\n".join(linhas_texto)

print("\nTexto gerado:")
print(texto_dre)



PASSO 3: Convertendo tabela em texto...

Texto gerado:
Petrobras (PETR4) — Demonstração do Resultado — 2023:

  Receita de Venda de Bens e/ou Serviços: R$ 511,994,000 mil
  Resultado Antes do Resultado Financeiro e dos Tributos: R$ 189,342,000 mil
  Resultado Financeiro: R$ -11,861,000 mil
  Despesas Financeiras: R$ -22,682,000 mil
  Lucro/Prejuízo Consolidado do Período: R$ 125,166,000 mil


In [4]:
# ============================================================
# PASSO 4: Criar Documents do LangChain
# ============================================================

"""
O que estamos fazendo
Empacotando o texto + metadados no formato que o LangChain espera.
Um Document é só um objeto com dois campos: page_content (texto)
e metadata (dicionário com informações extras).

De onde vem isso
LangChain docs (tutorial RAG): Depois do loading, os dados
viram objetos Document. É a unidade básica do LangChain.
Todo o pipeline trabalha com Documents.
"""
print("\n" + "=" * 50)
print("PASSO 4: Criando Documents do LangChain...")
print("=" * 50)

from langchain_core.documents import Document

# Vamos fazer o mesmo processo para todos os demonstrativos
# e criar um Document para cada um

DEMONSTRATIVOS = {
    "BPA": ("dfp_cia_aberta_BPA_con_2023.csv", "Balanço Patrimonial — Ativo"),
    "BPP": ("dfp_cia_aberta_BPP_con_2023.csv", "Balanço Patrimonial — Passivo"),
    "DRE": ("dfp_cia_aberta_DRE_con_2023.csv", "Demonstração do Resultado"),
    "DFC": ("dfp_cia_aberta_DFC_MI_con_2023.csv", "Fluxo de Caixa"),
}

# Contas relevantes por demonstrativo
CONTAS_POR_DEMO = {
    "BPA": ["1", "1.01", "1.01.01", "1.02"],
    "BPP": ["2", "2.01", "2.01.04", "2.02", "2.02.01", "2.03"],
    "DRE": ["3.01", "3.05", "3.06", "3.06.02", "3.11"],
    "DFC": ["6.01", "6.02", "6.03"],
}

documents = []

for demo_key, (filename, demo_name) in DEMONSTRATIVOS.items():
    # Carregar CSV
    df = pd.read_csv(
        z.open(filename),
        sep=";",
        encoding="latin-1",
        dtype={"CD_CONTA": str}
    )
    
    # Filtrar Petrobras + exercício mais recente
    df = df[df["CD_CVM"] == COD_CVM_PETROBRAS]
    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]
    
    # Pegar versão mais recente se houver duplicatas
    if "VERSAO" in df.columns:
        df = df[df["VERSAO"] == df["VERSAO"].max()]
    
    # Filtrar contas relevantes
    contas = CONTAS_POR_DEMO.get(demo_key, [])
    df = df[df["CD_CONTA"].isin(contas)]
    
    if df.empty:
        continue
    
    # Converter em texto
    linhas = [f"Petrobras (PETR4) — {demo_name} — Exercício 2023:", ""]
    for _, row in df.iterrows():
        valor = row["VL_CONTA"]
        conta = row["DS_CONTA"]
        linhas.append(f"  {conta}: R$ {valor:,.0f} mil")
    
    texto = "\n".join(linhas)
    
    # Criar Document
    doc = Document(
        page_content=texto,
        metadata={
            "company": "Petrobras",
            "ticker": "PETR4",
            "country": "BR",
            "document_type": "DFP",
            "section": demo_key.lower(),
            "fiscal_year": 2023,
            "source": "CVM",
        }
    )
    documents.append(doc)

print(f"\n{len(documents)} Documents criados:")
for doc in documents:
    preview = doc.page_content[:80] + "..."
    print(f"  [{doc.metadata['section']}] {preview}")


PASSO 4: Criando Documents do LangChain...

4 Documents criados:
  [bpa] Petrobras (PETR4) — Balanço Patrimonial — Ativo — Exercício 2023:

  Ativo Total...
  [bpp] Petrobras (PETR4) — Balanço Patrimonial — Passivo — Exercício 2023:

  Passivo T...
  [dre] Petrobras (PETR4) — Demonstração do Resultado — Exercício 2023:

  Receita de Ve...
  [dfc] Petrobras (PETR4) — Fluxo de Caixa — Exercício 2023:

  Caixa Líquido Atividades...


In [5]:
# ============================================================
# PASSO 5: Dividir em chunks
# ============================================================
"""
O que estamos fazendo
Dividindo os textos em pedaços menores. Por quê? Porque o retriever
precisa encontrar o PEDAÇO certo, não o documento inteiro. Se você
manda o balanço inteiro como um bloco só, e o usuário pergunta sobre
caixa, o retriever traz tudo (incluindo informação irrelevante).

De onde vem isso
LangChain docs (tutorial RAG): Depois do loading, o próximo
passo é "Split". O tutorial usa RecursiveCharacterTextSplitter.
Ele tenta quebrar por parágrafo, depois por frase, depois por
caractere — sempre respeitando o tamanho máximo do chunk.
Playlist RAG From Scratch (vídeo 2-3): Lance Martin explica
que o chunk_size e o overlap são os dois parâmetros mais importantes.
Chunk muito grande = muita informação irrelevante no contexto.
Chunk muito pequeno = perde contexto necessário.
Chip Huyen (Cap. 6 — RAG Architecture): A estratégia de
chunking impacta diretamente a qualidade do retrieval. Chunks
devem ser unidades semânticas coerentes.
"""
print("\n" + "=" * 50)
print("PASSO 5: Dividindo em chunks...")
print("=" * 50)

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Nossos documentos são curtos (dados tabulares convertidos),
# então usamos chunks menores que o padrão
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # cada chunk tem no máximo 500 caracteres
    chunk_overlap=50,     # 50 chars de sobreposição entre chunks
    separators=["\n\n", "\n", " "],  # tenta quebrar por parágrafo, linha, espaço
)

# Dividir todos os documents
chunks = splitter.split_documents(documents)

print(f"\n{len(documents)} Documents → {len(chunks)} chunks")
print("\nExemplo de chunk:")
print(f"  Texto: {chunks[0].page_content[:200]}...")
print(f"  Metadados: {chunks[0].metadata}")


PASSO 5: Dividindo em chunks...

4 Documents → 4 chunks

Exemplo de chunk:
  Texto: Petrobras (PETR4) — Balanço Patrimonial — Ativo — Exercício 2023:

  Ativo Total: R$ 1,050,888,000 mil
  Ativo Circulante: R$ 157,079,000 mil
  Caixa e Equivalentes de Caixa: R$ 61,613,000 mil
  Ativo...
  Metadados: {'company': 'Petrobras', 'ticker': 'PETR4', 'country': 'BR', 'document_type': 'DFP', 'section': 'bpa', 'fiscal_year': 2023, 'source': 'CVM'}


In [6]:
# ============================================================
# PASSO 6: Embeddings + Vector Store
# ============================================================

print("\n" + "=" * 50)
print("PASSO 6: Criando embeddings e armazenando...")
print("=" * 50)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Criar o modelo de embeddings
# text-embedding-3-small é barato e eficiente
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Criar o vector store a partir dos chunks
# Isso faz duas coisas:
# 1. Chama a API da OpenAI para gerar embedding de cada chunk
# 2. Armazena tudo no ChromaDB local
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",       # salva em disco
    collection_name="credit_risk_petrobras"
)

print(f"\n✅ {len(chunks)} chunks armazenados no ChromaDB")
print(f"   Diretório: ./chroma_db")

# Teste rápido: buscar chunks parecidos com uma pergunta
print("\n--- Teste de busca ---")
resultados = vectorstore.similarity_search(
    "Qual a dívida da Petrobras?",
    k=3  # retorna os 3 chunks mais relevantes
)

for i, doc in enumerate(resultados):
    print(f"\nResultado {i+1}:")
    print(f"  Seção: {doc.metadata['section']}")
    print(f"  Texto: {doc.page_content[:150]}...")


PASSO 6: Criando embeddings e armazenando...

✅ 4 chunks armazenados no ChromaDB
   Diretório: ./chroma_db

--- Teste de busca ---

Resultado 1:
  Seção: bpp
  Texto: Petrobras (PETR4) — Balanço Patrimonial — Passivo — Exercício 2023:

  Passivo Total: R$ 1,050,888,000 mil
  Passivo Circulante: R$ 163,928,000 mil
  ...

Resultado 2:
  Seção: bpp
  Texto: Petrobras (PETR4) — Balanço Patrimonial — Passivo — Exercício 2023:

  Passivo Total: R$ 1,050,888,000 mil
  Passivo Circulante: R$ 163,928,000 mil
  ...

Resultado 3:
  Seção: bpa
  Texto: Petrobras (PETR4) — Balanço Patrimonial — Ativo — Exercício 2023:

  Ativo Total: R$ 1,050,888,000 mil
  Ativo Circulante: R$ 157,079,000 mil
  Caixa ...


In [7]:
# ============================================================
# PASSO 7: Montar a chain de RAG
# ============================================================

print("\n" + "=" * 50)
print("PASSO 7: Montando a chain de RAG...")
print("=" * 50)

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# 1. O LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. O retriever (busca os chunks relevantes)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}  # retorna os 4 chunks mais relevantes
)

# 3. O prompt template
#    {context} será preenchido com os chunks encontrados
#    {question} será preenchido com a pergunta do usuário
prompt = ChatPromptTemplate.from_template("""
Você é um analista de risco de crédito especializado em empresas
brasileiras de capital aberto.

Responda a pergunta usando APENAS as informações do contexto abaixo.
Se não tiver informação suficiente no contexto, diga claramente.

Sempre cite a empresa, o período e a fonte dos dados.

Contexto:
{context}

Pergunta: {question}

Resposta:
""")

# 4. Função auxiliar: formata os chunks em texto para o prompt
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# 5. Montar a chain (LCEL - LangChain Expression Language)
#
#    Fluxo: pergunta → busca chunks → formata → monta prompt → LLM → texto
#
#    Isso é o que a playlist RAG From Scratch mostra no vídeo 1:
#    Query → Retrieve → Augment → Generate
#
rag_chain = (
    {
        "context": retriever | format_docs,    # busca e formata
        "question": RunnablePassthrough()       # passa a pergunta direto
    }
    | prompt       # monta o prompt com contexto + pergunta
    | llm          # envia pro LLM
    | StrOutputParser()  # extrai o texto da resposta
)

print("✅ Chain montada!")


PASSO 7: Montando a chain de RAG...
✅ Chain montada!


In [8]:
# ============================================================
# PASSO 8: Testar o RAG
# ============================================================

print("\n" + "=" * 50)
print("PASSO 8: Testando o RAG!")
print("=" * 50)

# Perguntas de teste para risco de crédito
perguntas = [
    "Qual o nível de endividamento da Petrobras em 2023?",
    "A Petrobras gera caixa operacional suficiente?",
    "Qual a liquidez corrente da Petrobras?",
    "Qual foi o lucro líquido da Petrobras em 2023?",
]

for pergunta in perguntas:
    print(f"\n{'─' * 50}")
    print(f"❓ {pergunta}")
    print(f"{'─' * 50}")
    
    resposta = rag_chain.invoke(pergunta)
    print(f"\n💬 {resposta}")


PASSO 8: Testando o RAG!

──────────────────────────────────────────────────
❓ Qual o nível de endividamento da Petrobras em 2023?
──────────────────────────────────────────────────

💬 Para calcular o nível de endividamento da Petrobras (PETR4) em 2023, podemos utilizar a relação entre o Passivo Total e o Patrimônio Líquido Consolidado.

O Passivo Total da Petrobras em 2023 é de R$ 1.050.888.000 mil e o Patrimônio Líquido Consolidado é de R$ 382.340.000 mil.

O nível de endividamento pode ser calculado pela fórmula:

\[ \text{Nível de Endividamento} = \frac{\text{Passivo Total}}{\text{Patrimônio Líquido}} \]

Substituindo os valores:

\[ \text{Nível de Endividamento} = \frac{1.050.888.000}{382.340.000} \approx 2,75 \]

Isso significa que para cada R$ 1,00 de patrimônio líquido, a Petrobras possui aproximadamente R$ 2,75 em dívidas.

Portanto, o nível de endividamento da Petrobras em 2023 é de aproximadamente 2,75. 

Fonte: Balanço Patrimonial da Petrobras (PETR4) - Exercício 2023.

──

In [9]:
# ============================================================
# PASSO 9: Debug — ver o que o retriever está trazendo
# ============================================================

print("\n" + "=" * 50)
print("PASSO 9: Inspecionando o retriever...")
print("=" * 50)

pergunta_teste = "A Petrobras tem muita dívida de curto prazo?"

# Buscar os chunks que o retriever retornaria
docs_encontrados = retriever.invoke(pergunta_teste)

print(f"\nPergunta: {pergunta_teste}")
print(f"Chunks encontrados: {len(docs_encontrados)}")

for i, doc in enumerate(docs_encontrados):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Seção: {doc.metadata['section']}")
    print(f"Ano: {doc.metadata['fiscal_year']}")
    print(f"Texto:\n{doc.page_content}")


PASSO 9: Inspecionando o retriever...

Pergunta: A Petrobras tem muita dívida de curto prazo?
Chunks encontrados: 4

--- Chunk 1 ---
Seção: bpp
Ano: 2023
Texto:
Petrobras (PETR4) — Balanço Patrimonial — Passivo — Exercício 2023:

  Passivo Total: R$ 1,050,888,000 mil
  Passivo Circulante: R$ 163,928,000 mil
  Empréstimos e Financiamentos: R$ 55,781,000 mil
  Passivo Não Circulante: R$ 504,620,000 mil
  Empréstimos e Financiamentos: R$ 247,281,000 mil
  Patrimônio Líquido Consolidado: R$ 382,340,000 mil

--- Chunk 2 ---
Seção: bpp
Ano: 2023
Texto:
Petrobras (PETR4) — Balanço Patrimonial — Passivo — Exercício 2023:

  Passivo Total: R$ 1,050,888,000 mil
  Passivo Circulante: R$ 163,928,000 mil
  Empréstimos e Financiamentos: R$ 55,781,000 mil
  Passivo Não Circulante: R$ 504,620,000 mil
  Empréstimos e Financiamentos: R$ 247,281,000 mil
  Patrimônio Líquido Consolidado: R$ 382,340,000 mil

--- Chunk 3 ---
Seção: bpa
Ano: 2023
Texto:
Petrobras (PETR4) — Balanço Patrimonial — Ativo — Exer